## Load Libraries

In [1]:
import pandas as pd
import numpy as np

## Helper Functions

In [2]:
column_translation = {
    "PÈrimËtre": "Region",
    "Nature": "Data_Type",
    "Date": "Date",
    "Heures": "Time",
    "Consommation": "Consumption_MW",
    "PrÈvision J-1": "Forecast_DayMinus1_MW",
    "PrÈvision J": "Forecast_Day_MW",
    "Fioul": "Oil_MW",
    "Charbon": "Coal_MW",
    "Gaz": "Gas_MW",
    "NuclÈaire": "Nuclear_MW",
    "Eolien": "Wind_MW",
    "Solaire": "Solar_MW",
    "Hydraulique": "Hydro_MW",
    "Pompage": "Pumped_Storage_MW",
    "BioÈnergies": "Bioenergy_MW",
    "Ech. physiques": "Physical_Exchange_MW",
    "Taux de Co2": "CO2_Rate_g_per_kWh",
    "Ech. comm. Angleterre": "Exchange_UK_MW",
    "Ech. comm. Espagne": "Exchange_Spain_MW",
    "Ech. comm. Italie": "Exchange_Italy_MW",
    "Ech. comm. Suisse": "Exchange_Switzerland_MW",
    "Ech. comm. Allemagne-Belgique": "Exchange_Germany_Belgium_MW",
    "Fioul - TAC": "Oil_TAC_MW",
    "Fioul - CogÈn.": "Oil_Cogeneration_MW",
    "Fioul - Autres": "Oil_Other_MW",
    "Gaz - TAC": "Gas_TAC_MW",
    "Gaz - CogÈn.": "Gas_Cogeneration_MW",
    "Gaz - CCG": "Gas_CCGT_MW",
    "Gaz - Autres": "Gas_Other_MW",
    "Hydraulique - Fil de l?eau + ÈclusÈe": "Hydro_RunOfRiver_MW",
    "Hydraulique - Lacs": "Hydro_Reservoir_MW",
    "Hydraulique - STEP turbinage": "Hydro_PumpedStorage_Generation_MW",
    "BioÈnergies - DÈchets": "Bioenergy_Waste_MW",
    "BioÈnergies - Biomasse": "Bioenergy_Biomass_MW",
    "BioÈnergies - Biogaz": "Bioenergy_Biogas_MW",
    " Stockage batterie": "Battery_Charging_MW",
    "DÈstockage batterie": "Battery_Discharging_MW",
    "Eolien terrestre": "Onshore_Wind_MW",
    "Eolien offshore": "Offshore_Wind_MW"
}


import pandas as pd
import numpy as np

def load_and_clean(file_path):

    # Load correctly
    df = pd.read_csv(file_path, sep=",", encoding="utf-8-sig")

    # Clean column names
    df.columns = df.columns.str.strip()

    # Rename columns
    df.rename(columns=column_translation, inplace=True)

    # Replace ND with NaN
    df.replace("ND", np.nan, inplace=True)

    # Keep only France
    df = df[df["Region"] == "France"]

    # Keep only final validated data
    df = df[df["Data_Type"].str.contains("Donn", na=False)]

    # Create datetime
    df["Datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        dayfirst=True,
        errors="coerce"
    )

    # Drop rows without consumption (removes :15 and :45 empty rows)
    df = df[df["Consumption_MW"].notna()]

    # ---- KEEP ONLY VARIABLES USEFUL FOR DEMAND PREDICTION ----
    df = df[[
        "Datetime",
        "Consumption_MW",
        "Forecast_DayMinus1_MW",
        "Forecast_Day_MW"
    ]]

    # Convert numeric
    df["Consumption_MW"] = pd.to_numeric(df["Consumption_MW"], errors="coerce")
    df["Forecast_DayMinus1_MW"] = pd.to_numeric(df["Forecast_DayMinus1_MW"], errors="coerce")
    df["Forecast_Day_MW"] = pd.to_numeric(df["Forecast_Day_MW"], errors="coerce")

    # Sort by time
    df = df.sort_values("Datetime").reset_index(drop=True)

    # TIME FEATURES
    df["Hour"] = df["Datetime"].dt.hour
    df["Minute"] = df["Datetime"].dt.minute
    df["DayOfWeek"] = df["Datetime"].dt.dayofweek
    df["Month"] = df["Datetime"].dt.month
    df["DayOfYear"] = df["Datetime"].dt.dayofyear
    df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

    # LAG FEATURES (15-min frequency)
    df = df.sort_values("Datetime")
    df["Lag_1"] = df["Consumption_MW"].shift(1)      # 15 min
    df["Lag_4"] = df["Consumption_MW"].shift(4)      # 1 hour
    df["Lag_96"] = df["Consumption_MW"].shift(96)    # 1 day
    df["Lag_672"] = df["Consumption_MW"].shift(672)  # 1 week

    # Drop rows where lags are NaN (first week will be removed)
    df = df.dropna(subset=["Lag_1", "Lag_4", "Lag_96", "Lag_672"]).reset_index(drop=True)

    return df

## Read all csv files and return a combined DataFrame

In [3]:
files = [
    "electricity/eCO2mix_RTE_Annuel-Definitif_2019.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2020.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2021.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2022.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2023.csv"
]

dfs = [load_and_clean(file) for file in files]
df_final = pd.concat(dfs, ignore_index=True)

# Save to CSV
df_final.to_csv("electricity/France_Electricity_Cleaned.csv", index=False, encoding="utf-8-sig")

df_final

/var/folders/w0/qsrp92ys7rb9d622jmmq_pfr0000gn/T/ipykernel_27755/3936007201.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace("ND", np.nan, inplace=True)
/var/folders/w0/qsrp92ys7rb9d622jmmq_pfr0000gn/T/ipykernel_27755/3936007201.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Datetime"] = pd.to_datetime(
/var/folders/w0/qsrp92ys7rb9d622jmmq_pfr0000gn/T/ipykernel_27755/3936007201.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavio

,Datetime,Consumption_MW,Forecast_DayMinus1_MW,Forecast_Day_MW,Hour,Minute,DayOfWeek,Month,DayOfYear,IsWeekend,Lag_1,Lag_4,Lag_96,Lag_672
0,2019-01-15 00:00:00,68453.0,68100.0,67400.0,0,0,1,1,15,0,68399.0,66594.0,68046.0,64207.0
1,2019-01-15 00:30:00,66512.0,66100.0,65000.0,0,30,1,1,15,0,68453.0,66733.0,66288.0,63162.0
2,2019-01-15 01:00:00,63738.0,63800.0,63000.0,1,0,1,1,15,0,66512.0,69446.0,63726.0,60923.0
3,2019-01-15 01:30:00,63478.0,64200.0,63600.0,1,30,1,1,15,0,63738.0,68399.0,62939.0,60729.0
4,2019-01-15 02:00:00,62867.0,64200.0,63000.0,2,0,1,1,15,0,63478.0,68453.0,62215.0,60127.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84283,2023-12-31 21:30:00,53329.0,50100.0,51900.0,21,30,6,12,365,1,53812.0,57547.0,53780.0,63045.0
84284,2023-12-31 22:00:00,52719.0,50600.0,52400.0,22,0,6,12,365,1,53329.0,56136.0,52793.0,61796.0
84285,2023-12-31 22:30:00,53518.0,51900.0,53700.0,22,30,6,12,365,1,52719.0,54956.0,53530.0,62386.0
84286,2023-12-31 23:00:00,55564.0,53000.0,54800.0,23,0,6,12,365,1,53518.0,53812.0,54835.0,63659.0
